# Baseline accuracy

Compare agent and patient next-token logits at the **last non-padding prompt token**. Accuracy means `agent_logit > patient_logit`; ties count as incorrect. This is pairwise preference, not unrestricted generation accuracy.

Use the repository's `.venv` kernel with the notebook dependencies installed (`uv sync --locked --group notebook`) and a GPU allocation for Pythia 6.9B in float32. The checkpoint must be cached or downloadable. No activation files are needed.

Run the model-loading cell once. To try another corpus or question, edit **Experiment settings** and rerun from that cell downward. Questions must still ask for the agent; candidates are the two nouns with a leading space, each occupying one token.

In [1]:
import csv
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from transformers import AutoTokenizer

from ac_tpr.dataset import CORPUS_COLUMNS, TOKENIZER_ID, TOKENIZER_REVISION
from ac_tpr.interventions import prepare_prompts, score_batch
from ac_tpr.model import load_model
import os
os.environ["DISABLE_SAFETENSORS_CONVERSION"] = "1"

ROOT = Path.cwd()
if ROOT.name == "notebook":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "Run from the repository root or notebook directory"


/u/dssc/zenocosini/AC-TPR-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the model once

In [2]:
DEVICE = "cuda"
DTYPE = "float32"

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID, revision=TOKENIZER_REVISION, use_fast=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model, model_metadata = load_model(tokenizer, TOKENIZER_REVISION, DEVICE, DTYPE)
print(model_metadata)


Loading weights: 100%|██████████| 388/388 [00:03<00:00, 125.10it/s]


{'name': 'EleutherAI/pythia-6.9b', 'revision': 'step143000', 'commit': '21bfa02e806e253fe453702c29c81d9f83617255', 'dtype': 'float32', 'attention_implementation': 'eager', 'backend': 'nnsight', 'api': 'LanguageModel', 'nnsight_version': '0.7.0', 'transformers_version': '5.9.0', 'weight_processing': 'none'}


## Experiment settings

Keep `{sentence}` at the start of the template. The default ends at `Answer:` with no trailing whitespace. New CSVs use the existing corpus schema. Set `SPLITS = None` to use every split; `LIMIT` selects the first matching rows in corpus order.

In [8]:
CORPUS = ROOT / "data/generated/corpus.csv"
SPLITS = ["test", "gen_test"]
PROMPT_TEMPLATE = "{sentence}\nQuestion: Who is performing the action (answer only with the name of the person)?\nAnswer:"
BATCH_SIZE = 32
LIMIT = None


In [9]:
assert BATCH_SIZE > 0
assert LIMIT is None or LIMIT > 0
with CORPUS.open(newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    assert reader.fieldnames == CORPUS_COLUMNS, f"Expected columns: {CORPUS_COLUMNS}"
    corpus = list(reader)
if SPLITS is not None:
    assert set(SPLITS) <= {row["split"] for row in corpus}, "Requested split is absent"
rows = [row for row in corpus if SPLITS is None or row["split"] in SPLITS][:LIMIT]
assert rows, "No rows selected"

input_ids, role_positions, answer_ids = prepare_prompts(
    rows, tokenizer, prompt_template=PROMPT_TEMPLATE
)
assert max(map(len, input_ids)) <= model.config.max_position_embeddings, "Prompt exceeds model context length"
prompts = [PROMPT_TEMPLATE.format(sentence=row["sentence"]) for row in rows]
print(f"{len(rows):,} examples from {CORPUS}")
display(pd.DataFrame(rows).groupby(["split", "voice"]).size().rename("n").to_frame())


1,940 examples from /orfeo/cephfs/home/dssc/zenocosini/AC-TPR-project/data/generated/corpus.csv


n
split    voice       
gen_test active   864
         passive  864
test     active   106
         passive  106

## Inspect the prompt

In [10]:
EXAMPLE = 0
print(prompts[EXAMPLE])
print("\nExact prompt:", repr(prompts[EXAMPLE]))
print("Last prompt token:", repr(tokenizer.decode([input_ids[EXAMPLE][-1]])))
for role, token_id in zip(["agent", "patient"], answer_ids[EXAMPLE].tolist()):
    print(f"{role}: {tokenizer.decode([token_id])!r} (token ID {token_id})")


The actor called the artist.
Question: Who is performing the action (answer only with the name of the person)?
Answer:

Exact prompt: 'The actor called the artist.\nQuestion: Who is performing the action (answer only with the name of the person)?\nAnswer:'
Last prompt token: ':'
agent: ' actor' (token ID 12353)
patient: ' artist' (token ID 9558)


## Generate an answer

Run after loading the model and experiment settings. Paste a dataset sentence below (or use `rows[EXAMPLE]["sentence"]`), then rerun this cell. Generation uses the same `PROMPT_TEMPLATE` as the baseline and stops at EOS or 64 new tokens.

In [ ]:
prompt

'The actor called the artist.\nQuestion: Who is performing the action (answer only with the name of the person)?\nAnswer:'

: 

In [13]:
SENTENCE = rows[EXAMPLE]["sentence"]  # Or paste a dataset sentence here.
prompt = PROMPT_TEMPLATE.format(sentence=SENTENCE)
inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(DEVICE)

with torch.inference_mode():
    output = model._model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))

 The actor called the artist.

A:

The correct answer is:

The actor called the artist.

The actor is performing the action, and the artist is the person being acted upon.
The actor is performing the action, and the artist is the person performing the action.
The


## Score the baseline

The existing scorer selects the final non-padding position using the attention mask. No role edits are applied.

In [11]:
batches = []
for start in range(0, len(rows), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(rows))
    inputs = tokenizer.pad(
        {"input_ids": input_ids[start:end]}, padding=True, return_tensors="pt"
    ).to(DEVICE)
    batches.append(score_batch(model, inputs, answer_ids[start:end]))
    print(f"Scored {end:,}/{len(rows):,}", end="\r")
print()
scores = torch.cat(batches).numpy()

results = pd.DataFrame(rows)
results["prompt"] = prompts
results["agent_logit"] = scores[:, 0]
results["patient_logit"] = scores[:, 1]
results["logit_diff"] = results["patient_logit"] - results["agent_logit"]
results["correct"] = results["agent_logit"] > results["patient_logit"]
results["patient_wins"] = results["patient_logit"] > results["agent_logit"]
results["tie"] = results["agent_logit"] == results["patient_logit"]


Scored 1,940/1,940


## Results

Accuracy uses all selected examples, including ties. Positive logit differences favor the patient. The full per-example table remains available as `results`.

In [12]:
display(pd.DataFrame([{
    "n": len(results),
    "accuracy": results["correct"].mean(),
    "patient_preference_rate": results["patient_wins"].mean(),
    "ties": int(results["tie"].sum()),
    "mean_logit_diff": results["logit_diff"].mean(),
}], index=["overall"]))

display(results.groupby(["split", "voice"]).agg(
    n=("correct", "size"),
    accuracy=("correct", "mean"),
    patient_preference_rate=("patient_wins", "mean"),
    ties=("tie", "sum"),
    mean_logit_diff=("logit_diff", "mean"),
))

print("Examples with the largest patient preference:")
display(
    results.loc[results["patient_wins"]]
    .sort_values("logit_diff", ascending=False)
    .head(20)[[
        "sentence", "split", "voice", "agent", "patient",
        "agent_logit", "patient_logit", "logit_diff",
    ]]
)


,n,accuracy,patient_preference_rate,ties,mean_logit_diff
overall,1940,0.543814,0.456186,0,-0.208783


n  accuracy  patient_preference_rate  ties  \
split    voice                                                   
gen_test active   864  0.489583                 0.510417     0   
         passive  864  0.596065                 0.403935     0   
test     active   106  0.471698                 0.528302     0   
         passive  106  0.632075                 0.367925     0   

                  mean_logit_diff  
split    voice                     
gen_test active         -0.030458  
         passive        -0.367265  
test     active          0.009769  
         passive        -0.589062

Examples with the largest patient preference:


,sentence,split,voice,agent,patient,agent_logit,patient_logit,logit_diff
1924,The woman called the teacher.,gen_test,active,woman,teacher,4.613618,9.342612,4.728994
447,The doctor was liked by the boy.,test,passive,boy,doctor,4.834195,9.304103,4.469908
579,The doctor was loved by the child.,gen_test,passive,child,doctor,4.762606,9.184189,4.421583
498,The boy watched the teacher.,gen_test,active,boy,teacher,4.810560,9.142330,4.331770
495,The teacher was loved by the boy.,gen_test,passive,boy,teacher,4.681121,8.927166,4.246045
577,The doctor was liked by the child.,gen_test,passive,child,doctor,5.420234,9.611115,4.190880
1938,The woman watched the teacher.,gen_test,active,woman,teacher,4.315574,8.497232,4.181658
449,The doctor was touched by the boy.,test,passive,boy,doctor,4.208393,8.383634,4.175241
942,The girl helped the doctor.,test,active,girl,doctor,5.384541,9.502786,4.118245
582,The child watched the doctor.,gen_test,active,child,doctor,5.809908,9.855187,4.045280
